<div align="center">

# **Estudio de Patrones de Movilidad Urbana mediante Cadenas de Markov**

---

## **Actividad Colaborativa**

### **Modelado Probabilístico de Movilidad Urbana usando Cadenas de Markov**

<br><br>

### **Integrantes del equipo**

#### Fernando Mata Martinez



<br><br>

### **Curso**

#### Metodos Estocasticos

### **Docente**

#### Jose Lazaro Martinez Rodriguez

### **Grupo**

#### G

### **Institución**

#### Universidad Autonoma de Tamaulipas

<br><br><br>

### **Fecha de entrega**

####

---

## **Objetivo de la actividad**

Analizar patrones de movilidad urbana mediante el uso de cadenas de Markov, construyendo una matriz de transición a partir de observaciones reales y estudiando el comportamiento probabilístico y estacionario del sistema utilizando herramientas computacionales en Python.

<br><br>

### **Enlace al video explicativo**

####

</div>


In [2]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

# Configurar semilla para que la simulación sea reproducible
np.random.seed(42)

# Enlace RAW del archivo transporte_urbano en GitHub
url = 'https://raw.githubusercontent.com/fermatta93/transporte_urbano/refs/heads/main/transporte_urbano%20(1).csv'

# Descargar y leer directamente
df = pd.read_csv(url)

print(df.head())
print(f'Éxito: se cargaron {len(df)} registros.')

   paso            estado
0     1              Casa
1     2           Trabajo
2     3            Parque
3     4  Centro_Comercial
4     5           Trabajo
Éxito: se cargaron 10000 registros.


## Parte I. Exploración de los datos

Se carga el archivo y se obtienen el número de registros y los estados disponibles y frecuencias.

**Interpretación:** el ciudadano pasa la mayor parte de su tiempo en **Casa** (39.53%) y **Trabajo** (32.09%), que juntos concentran más del 71% de las observaciones. Centro Comercial, Gimnasio y Parque son destinos secundarios, con **Parque** como el menos frecuentado (7.70%).



In [3]:

# PARTE I: EXPLORACIÓN DE LOS DATOS

estados_orden = ['Casa', 'Trabajo', 'Centro_Comercial', 'Gimnasio', 'Parque']

print(f"Número total de registros: {len(df)}")
print(f"Estados disponibles: {sorted(df['estado'].unique())}")

frecuencias = df['estado'].value_counts().reindex(estados_orden)
proporciones = df['estado'].value_counts(normalize=True).reindex(estados_orden)

tabla_exploracion = pd.DataFrame({
    'Frecuencia Absoluta': frecuencias,
    'Proporcion (%)': (proporciones * 100).round(2)
})
print("\nTabla de frecuencias:")
print(tabla_exploracion)


Número total de registros: 10000
Estados disponibles: ['Casa', 'Centro_Comercial', 'Gimnasio', 'Parque', 'Trabajo']

Tabla de frecuencias:
                  Frecuencia Absoluta  Proporcion (%)
estado                                               
Casa                             3953           39.53
Trabajo                          3209           32.09
Centro_Comercial                 1046           10.46
Gimnasio                         1022           10.22
Parque                            770            7.70


## Parte II. Matriz de transición

A partir de los pares consecutivos $(X_n, X_{n+1})$ se construye la tabla de conteos y se normaliza por fila para obtener la matriz de transición estocástica $P=[p_{ij}]$, donde $p_{ij}=P(X_{n+1}=j \mid X_n=i)$.

La verificación confirma que cada fila de $P$ suma exactamente 1, como exige una matriz estocástica.

In [4]:
# PARTE II: MATRIZ DE TRANSICIÓN
secuencia = df['estado'].values
transiciones = pd.DataFrame({'actual': secuencia[:-1], 'siguiente': secuencia[1:]})

# Tabla de conteos de transiciones (X_n, X_n+1)
tabla_conteos = pd.crosstab(transiciones['actual'], transiciones['siguiente']) \
    .reindex(index=estados_orden, columns=estados_orden).fillna(0).astype(int)
print("Tabla de conteos de transiciones:")
print(tabla_conteos)

# Estimación de la matriz de transición P
P = tabla_conteos.div(tabla_conteos.sum(axis=1), axis=0).values
P_df = pd.DataFrame(P, index=estados_orden, columns=estados_orden)
print("\nMatriz de transición estimada P:")
print(P_df.round(6))

print("\nVerificación: suma de cada fila =", P.sum(axis=1).round(6))


Tabla de conteos de transiciones:
siguiente         Casa  Trabajo  Centro_Comercial  Gimnasio  Parque
actual                                                             
Casa               604     2561               325       255     207
Trabajo           1833      306               460       480     130
Centro_Comercial   477      168                97       101     203
Gimnasio           547      100               102       114     159
Parque             491       74                62        72      71

Matriz de transición estimada P:
                      Casa   Trabajo  Centro_Comercial  Gimnasio    Parque
Casa              0.152834  0.648026          0.082237  0.064524  0.052379
Trabajo           0.571206  0.095357          0.143347  0.149579  0.040511
Centro_Comercial  0.456023  0.160612          0.092734  0.096558  0.194073
Gimnasio          0.535225  0.097847          0.099804  0.111546  0.155577
Parque            0.637662  0.096104          0.080519  0.093506  0.092208

Verif

## Parte III. Ecuaciones de Chapman-Kolmogorov

Usando $P^2$, $P^3$ y $P^{10}$ se responde:

1. **P(Casa → Parque en 2 pasos) = 0.065086 (6.51%)** — entrada $[Casa, Parque]$ de $P^2$, que suma todos los caminos de longitud 2 vía cualquier estado intermedio $k$: $\sum_k p_{Casa,k}\, p_{k,Parque}$.
2. **P(Trabajo → Gimnasio en 3 pasos) = 0.109852 (10.99%)** — entrada $[Trabajo, Gimnasio]$ de $P^3$.
3. **P(Casa → Casa en 10 pasos) = 0.395422 (39.54%)** — entrada $[Casa, Casa]$ de $P^{10}$. Este valor ya es prácticamente idéntico a la probabilidad límite de Casa, lo que indica una convergencia rápida.

In [5]:
# PARTE III: ECUACIONES DE CHAPMAN-KOLMOGOROV

P2 = np.linalg.matrix_power(P, 2)
P3 = np.linalg.matrix_power(P, 3)
P10 = np.linalg.matrix_power(P, 10)

idx = {estado: i for i, estado in enumerate(estados_orden)}

prob_casa_parque_2 = P2[idx['Casa'], idx['Parque']]
prob_trabajo_gym_3 = P3[idx['Trabajo'], idx['Gimnasio']]
prob_casa_casa_10  = P10[idx['Casa'], idx['Casa']]

print(f"P(Casa -> Parque en 2 pasos)   = {prob_casa_parque_2:.6f}  ({prob_casa_parque_2*100:.2f}%)")
print(f"P(Trabajo -> Gimnasio en 3 pasos) = {prob_trabajo_gym_3:.6f}  ({prob_trabajo_gym_3*100:.2f}%)")
print(f"P(Casa -> Casa en 10 pasos)    = {prob_casa_casa_10:.6f}  ({prob_casa_casa_10*100:.2f}%)")


P(Casa -> Parque en 2 pasos)   = 0.065086  (6.51%)
P(Trabajo -> Gimnasio en 3 pasos) = 0.109852  (10.99%)
P(Casa -> Casa en 10 pasos)    = 0.395422  (39.54%)


1.¿Cuál es la probabilidad de que una persona que hoy está en Casa se encuentre en Parque dentro de 2 pasos?

Respuesta = 0.065086  (6.51%)

2.¿Cuál es la probabilidad de que una persona que hoy está en Trabajo se encuentre en Gimnasio dentro de 3 pasos?

Respuesta = 0.109852 (10.99%)

3.¿Cuál es la probabilidad de regresar a Casa después de 10 pasos si
actualmente se encuentra en Casa?

Respuesta = 0.395422 (39.54%)